# InciGraph quickstart

This notebook walks through the InciGraph Python API. By the end of it you will:

- know what data is available and how to find a sequence
- pull a tidy slice of estimates for any sequence and stratification
- pool across multiple age bands with the confidence intervals recomputed correctly
- compute a crude incidence-rate-ratio between two demographic groups, with its 95% CI and Wald p-value

The whole API is seven public functions. Each cell below introduces one.

## Setup

InciGraph reads from a directory of parquet files. Set it once at the top of your notebook:

In [ ]:
import incigraph as ig

ig.set_data_dir("./incigraph_data")  # or wherever your parquet files live
ig.__version__

## 1. What stratifications are available?

Estimates are precomputed across 15 stratification schemes. `available_stratifications()` returns the canonical keys you'll pass to `get_sequence`:

In [ ]:
ig.available_stratifications()

Read these as `"DIMENSION1+DIMENSION2+..."` in alphabetical order. `"NONE"` is the unstratified tree. `"ETHNICITY+IMD"` is two-way intersectional by ethnicity and deprivation. `"AGE_CATG+ETHNICITY+SEX"` is three-way. `AGE_CATG` is the age band, since it can be pooled at query time.

## 2. Which sequences exist?

`load_metadata()` returns a small lookup table of every sequence in the data:

In [ ]:
meta = ig.load_metadata()
print(f"{len(meta):,} unique sequences in the metadata table")
meta.head()

Each row has `sequence` (the canonical string like `"0 3 8"` with the leading cohort-root `0`), the trajectory `sequence_decoded` (e.g. `"HYPERTENSION -> T2D"`), and the per-step short disease names.

Filtering the metadata is the way to find sequences for your question:

In [ ]:
# What two-step sequences start with hypertension?
ig.list_sequences(starts_with_disease="HYPERTENSION", sequence_length=2).head(10)

In [ ]:
# What three-step sequences end with COPD?
ig.list_sequences(ends_with_disease="COPD", sequence_length=3).head(10)

## 3. Pull one slice: `get_sequence`

`get_sequence(sequence, stratification, age_bands=None)` is the headline function. It returns a tidy DataFrame ready for plotting:

In [ ]:
# Incidence of T2D after hypertension, by ethnicity and deprivation, ages 51-60
df = ig.get_sequence(
    sequence="0 3 8",                       # hypertension -> T2D
    stratification="AGE_CATG+ETHNICITY+IMD",
    age_bands=["51-60"],
)
df.head(12)

The columns to know about:

- `numerator`, `denominator`: incident events and person-years at risk
- `incidence_rate`: per 100,000 person-years
- `lower_limit`, `upper_limit`: 95% CI bounds (chi-squared for N<10, Byar's for N≥10)
- `imd_missing`: `True` for the IMD-missing stratum (a real subgroup, not a pooled total)
- `sex == "I"`: a real demographic category, not a sex-pooled total

**Filtering note.** The IMD-missing and `SEX='I'` rows are included by default because they are real strata. If your analysis needs only the IMD 1-5 quintiles or only M/F, filter them out explicitly:

In [ ]:
df_clean = df[~df["imd_missing"] & df["sex"].isin(["M", "F"])]
# or: df[df.imd.between(1, 5) & df.sex.isin(['M', 'F'])]
df_clean.shape

## 4. Pool age bands

Passing two or more bands to `age_bands` pools them. Numerators and person-time are summed within each (other-stratum) cell, then the rate and 95% CI are recomputed from the cumulative count using the same chi-squared/Byar's rule. This is the only correct way to combine age bands; never average incidence rates.

In [ ]:
# Pool ages 51-60 and 61-70 into a 51-70 cohort
df_pool = ig.get_sequence(
    sequence="0 3 8",
    stratification="AGE_CATG+ETHNICITY+IMD",
    age_bands=["51-60", "61-70"],
)
# the age_catg column is now NA because the rows are pooled across bands
df_pool.head(6)

## 5. Compute an IRR

`compute_irr(df, comparison, reference)` returns a crude IRR with its 95% Wald CI, log-IRR, standard error, and a two-sided Wald p-value. The two filter dicts pick one row each from `df`:

In [ ]:
# South Asian (most deprived) vs White (least deprived), within the 51-70 cohort
result = ig.compute_irr(
    df_pool,
    comparison={"ethnicity": "SOUTH_ASIAN", "imd": 5.0},
    reference={"ethnicity": "WHITE", "imd": 1.0},
)
for k, v in result.items():
    print(f"  {k:<20s}  {v}")

**A few cautions, baked into the response letter to any reviewer.** The SE formula assumes independent Poisson counts and is adequate for N ≥ 30 in both groups. Crude IRRs are not adjusted for confounding, competing risks, or differential coding by demographic group. The p-value is raw Wald — do not present it as confirmatory inference; if you are running many contrasts, apply a screening adjustment such as Benjamini-Hochberg.

## 6. The decoder, and a quick chain

`decode_sequence` is a small convenience for labels and plots:

In [ ]:
ig.decode_sequence("0 3 8 9")  # the cardiometabolic cascade

## 7. The raw escape hatch: `load_estimates`

If you need to do something the API doesn't directly support, `load_estimates(sequence_length)` gives you the full long-form DataFrame for one length. Use this if you want to write your own queries with pandas:

In [ ]:
# Pull just length-2 estimates and find the largest deprivation contrasts
L2 = ig.load_estimates(2)
L2.shape

Avoid `load_estimates(None)` for routine work — it concatenates all three lengths and may use several GB of memory.

## Where to go next

- `02_replicate_hero_figure.ipynb` — rebuilds the main manuscript figure (arithmetic check + selected contrasts + sparsity heatmap) using only the API.
- `03_replicate_l3_figure.ipynb` — rebuilds the length-3 convergence figure.
- The data dictionary in `docs/data_dictionary.md` documents every column in the parquet files.
- The `scripts/` directory contains command-line tools that wrap this same API.

## Quick reference

| Function | What it does |
|---|---|
| `set_data_dir(path)` | Tell the library where the parquet files live (call once per session) |
| `available_stratifications()` | List the 15 stratification keys |
| `load_metadata()` | The sequence lookup table |
| `list_sequences(...)` | Filter the metadata to find sequences |
| `get_sequence(seq, strat, age_bands=...)` | Tidy slice for one sequence + stratification |
| `compute_irr(df, comparison, reference)` | Crude IRR with 95% CI and Wald p-value |
| `decode_sequence(seq)` | `'0 3 8'` -> `'HYPERTENSION -> T2D'` |
| `load_estimates(length)` | Raw long-form DataFrame for the requested length |